In [ ]:
!pip install --upgrade pip setuptools wheel modelscope
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [3]:
!pip install lpips opencv-python scikit-image numpy 

In [ ]:
!pip install -r requirements312.txt

In [ ]:
import torch
torch.__version__
!git config --global --add safe.directory /home/h/DDColor

In [ ]:
!pip install -e . --no-build-isolation
#!python setup.py develop

In [ ]:
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download('damo/cv_ddcolor_image-colorization', cache_dir='./modelscope')
print('model assets saved to %s' % model_dir)

In [ ]:
torch.cuda.is_available()

In [3]:
# chuan bi data
import os
import pandas as pd
import shutil
from tqdm import tqdm

def prepare_data(csv_file, split_name):
    # Tên folder gốc chứa ảnh của bạn
    base_dir = 'ViCoW_Dataset' 
    
    if not os.path.exists(csv_file):
        print(f"Bỏ qua: Không tìm thấy file {csv_file}")
        return

    # Tạo thư mục đích: dataset/train, dataset/val, dataset/test
    output_dir = os.path.join('dataset', split_name)
    os.makedirs(output_dir, exist_ok=True)

    # Đọc CSV
    df = pd.read_csv(csv_file)
    print(f"\n--- Đang xử lý tập {split_name.upper()} ({len(df)} ảnh) ---")
    
    success_count = 0
    missing_samples = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Đường dẫn từ CSV: Color_image/VIDEO2_VaoNamRaBac/frame_0998.jpg
        rel_path = row['colorPath']
        
        # Ghép thành đường dẫn thực tế: ViCoW_Dataset/Color_image/...
        src_path = os.path.join(base_dir, rel_path)
        
        if os.path.exists(src_path):
            # Tách lấy tên VIDEO và tên FRAME để tạo tên file mới (tránh trùng)
            # Ví dụ: VIDEO2_VaoNamRaBac_frame_0998.jpg
            path_parts = rel_path.split('/')
            video_folder = path_parts[1] 
            filename = path_parts[-1]
            new_filename = f"{video_folder}_{filename}"
            
            dst_path = os.path.join(output_dir, new_filename)
            shutil.copy2(src_path, dst_path)
            success_count += 1
        else:
            if len(missing_samples) < 1: # Lưu lại mẫu lỗi đầu tiên để báo cáo
                missing_samples.append(os.path.abspath(src_path))

    print(f"Kết quả: Copy thành công {success_count}/{len(df)} ảnh vào '{output_dir}'")
    
    if success_count == 0 and len(missing_samples) > 0:
        print(f"\n[CẢNH BÁO] Không tìm thấy ảnh nào! Script đã thử tìm ở:")
        print(f" -> {missing_samples[0]}")
        print("Hãy đảm bảo folder 'ViCoW_Dataset' nằm cùng cấp với file script này.")

# --- Chạy script ---
tasks = [('ViCoW_Dataset/train.csv', 'train'), ('ViCoW_Dataset/val.csv', 'val'), ('ViCoW_Dataset/test.csv', 'test')]
for csv, split in tasks:
    prepare_data(csv, split)

print("\nHoàn tất!")


--- Đang xử lý tập TRAIN (1327 ảnh) ---


100%|██████████| 1327/1327 [00:01<00:00, 665.50it/s]


Kết quả: Copy thành công 1327/1327 ảnh vào 'dataset/train'

--- Đang xử lý tập VAL (187 ảnh) ---


100%|██████████| 187/187 [00:00<00:00, 635.71it/s]


Kết quả: Copy thành công 187/187 ảnh vào 'dataset/val'

--- Đang xử lý tập TEST (382 ảnh) ---


100%|██████████| 382/382 [00:00<00:00, 616.32it/s]

Kết quả: Copy thành công 382/382 ảnh vào 'dataset/test'

Hoàn tất!


In [7]:
%%bash
python data_list/get_meta_file.py --output-name ./dataset/train.txt --data-path ./dataset/train
python data_list/get_meta_file.py --output-name ./dataset/val.txt --data-path ./dataset/val
python data_list/get_meta_file.py --output-name ./dataset/test.txt --data-path ./dataset/test

Generating ./dataset/train.txt from ./dataset/train ...


100%|██████████| 1327/1327 [00:00<00:00, 2586357.53it/s]


Done.
Generating ./dataset/val.txt from ./dataset/val ...


100%|██████████| 187/187 [00:00<00:00, 1415766.87it/s]


Done.
Generating ./dataset/test.txt from ./dataset/test ...


100%|██████████| 382/382 [00:00<00:00, 2083516.42it/s]


Done.


In [ ]:
!wget https://dl.fbaipublicfiles.com/convnext/convnext_large_22k_224.pth -P pretrain/
!wget https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth -P pretrain/

In [1]:
# /home/h/DDColor/options/train/train_ViCoW.yml
!sh scripts/train.sh

/home/h/ddcolor312/lib/python3.12/site-packages/torch/distributed/launch.py:208: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  main()
Set pretrain_network_g to /home/h/DDColor/experiments/train_ViCoW_2/models/net_g_20000.pth
Set pretrain_network_d to /home/h/DDColor/experiments/train_ViCoW_2/models/net_d_20000.pth
2025-12-30 19:55:25,919 INFO: 
                ____                _       _____  ____
               / __ ) ____ _ _____ (_)_____/ ___/ / __ \
              / __  |/ __ `// ___// // ___/\__ \ / /_/ /
             / /_/ // /_/ /(__  )/ // /__ ___/ // _, _/
            /_____/ \__,_//____//_/ \___//____//_/ |_|
     ______                   __   

### Tensorboard

In [8]:
!tensorboard --logdir=tb_logger/train_ViCoW_2 --port=6006

/home/h/ddcolor312/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.20.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C


## Infer, gen out_test

In [ ]:
# !python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ./assets/test_images

In [9]:
#!python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512 --model_size large
#DDColor/experiments/train_ViCoW/models/net_g_5000.pth

!python infer.py --model_path experiments/train_ViCoW_2/models/net_g_25000.pth \
                --input dataset/test   \
                --output out_test_25000 \
                --input_size 512 # --model_size large

# from infer_hf import DDColorHF

# ddcolor_paper_tiny = DDColorHF.from_pretrained("piddnad/ddcolor_paper_tiny")
# ddcolor_paper      = DDColorHF.from_pretrained("piddnad/ddcolor_paper")
# ddcolor_modelscope = DDColorHF.from_pretrained("piddnad/ddcolor_modelscope")
# ddcolor_artistic   = DDColorHF.from_pretrained("piddnad/ddcolor_artistic")

# python infer_hf.py --model_name ddcolor_artistic --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out_artistic --input_size 512

# python infer_hf.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512


/home/h/ddcolor312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Output path: out_test_15000
100%|█████████████████████████████████████████| 382/382 [00:45<00:00,  8.41it/s]


## Validate

In [10]:
import os
import cv2
import torch
import numpy as np
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Khởi tạo LPIPS (VGG) - Yêu cầu: pip install lpips torch
loss_fn_vgg = lpips.LPIPS(net='vgg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_vgg.to(device)

def get_colorfulness(image):
    """Tính độ rực rỡ của ảnh (Hasler and Suesstrunk)"""
    (B, G, R) = cv2.split(image.astype("float"))
    rg = np.absolute(R - G)
    yb = np.absolute(0.5 * (R + G) - B)
    std_root = np.sqrt(np.std(rg)**2 + np.std(yb)**2)
    mean_root = np.sqrt(np.mean(rg)**2 + np.mean(yb)**2)
    return std_root + (0.3 * mean_root)

def im2tensor(image):
    """Chuyển ảnh OpenCV sang Tensor cho LPIPS"""
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = torch.from_numpy(image).permute(2, 0, 1).float()
    image = (image / 127.5) - 1.0
    return image.unsqueeze(0).to(device)

def calculate_full_metrics(gt_dir, out_dir):
    gt_images = sorted([f for f in os.listdir(gt_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    # Lưu trữ kết quả
    m = {'psnr': [], 'ssim': [], 'lpips': [], 'c_ratio': []}
    
    print(f"--- Đang đánh giá {len(gt_images)} ảnh trên {device} ---")

    for filename in gt_images:
        path_gt = os.path.join(gt_dir, filename)
        path_out = os.path.join(out_dir, filename)

        if not os.path.exists(path_out): continue

        img_gt = cv2.imread(path_gt)
        img_out = cv2.imread(path_out)

        # Đảm bảo cùng kích thước
        if img_gt.shape != img_out.shape:
            img_out = cv2.resize(img_out, (img_gt.shape[1], img_gt.shape[0]))

        # 1. PSNR & SSIM (Độ khớp pixel & cấu trúc)
        m['psnr'].append(psnr(img_gt, img_out, data_range=255))
        m['ssim'].append(ssim(img_gt, img_out, channel_axis=2, data_range=255))

        # 2. LPIPS (Độ chân thực cảm quan)
        t_gt, t_out = im2tensor(img_gt), im2tensor(img_out)
        with torch.no_grad():
            m['lpips'].append(loss_fn_vgg(t_gt, t_out).item())

        # 3. Colorfulness Ratio (Độ đậm nhạt của màu)
        c_gt = get_colorfulness(img_gt)
        c_out = get_colorfulness(img_out)
        m['c_ratio'].append((c_out / c_gt) * 100 if c_gt != 0 else 100)

        print(f"[{filename}] PSNR: {m['psnr'][-1]:.2f} | SSIM: {m['ssim'][-1]:.4f} | LPIPS: {m['lpips'][-1]:.4f} | Color: {m['c_ratio'][-1]:.1f}%")

    if m['psnr']:
        print("\n" + "="*50)
        print(f"KẾT QUẢ TRUNG BÌNH TOÀN BỘ TẬP TEST:")
        print(f"1. PSNR (Độ khớp pixel)   : {np.mean(m['psnr']):.2f} dB ↑")
        print(f"2. SSIM (Độ khớp cấu trúc): {np.mean(m['ssim']):.4f} ↑")
        print(f"3. LPIPS (Độ chân thực AI): {np.mean(m['lpips']):.4f} ↓ (Thấp là tốt)")
        print(f"4. COLOR RATIO (Độ đậm)   : {np.mean(m['c_ratio']):.1f}% (Càng gần 100% càng tốt)")
        print("="*50)
        
        # Đưa ra chẩn đoán
        avg_c = np.mean(m['c_ratio'])
        if avg_c < 85:
            print("CHẨN ĐOÁN: Ảnh đang bị NHẠT. Hãy tăng color_enhance_factor trong config.")
        elif avg_c > 115:
            print("CHẨN ĐOÁN: Ảnh đang bị QUÁ RỰC. Hãy giảm bớt color_enhance_factor.")
        else:
            print("CHẨN ĐOÁN: Màu sắc đã đạt độ bão hòa tương đồng với ảnh gốc.")
    else:
        print("Không có dữ liệu.")

if __name__ == "__main__":
    folder_gt = "dataset/test"
    folder_out = "out_test_25000"
    calculate_full_metrics(folder_gt, folder_out)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/h/ddcolor312/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/h/ddcolor312/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/h/ddcolor312/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth
--- Đang đánh giá 382 ảnh trên cuda ---


/home/h/ddcolor312/lib/python3.12/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path, map_location='c

[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0095.jpg] PSNR: 24.39 | SSIM: 0.9489 | LPIPS: 0.3065 | Color: 51.6%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0115.jpg] PSNR: 26.08 | SSIM: 0.9606 | LPIPS: 0.3086 | Color: 83.8%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0126.jpg] PSNR: 28.68 | SSIM: 0.9542 | LPIPS: 0.1838 | Color: 59.9%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0131.jpg] PSNR: 28.50 | SSIM: 0.9598 | LPIPS: 0.1804 | Color: 45.1%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0151.jpg] PSNR: 23.11 | SSIM: 0.9538 | LPIPS: 0.2396 | Color: 82.0%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0152.jpg] PSNR: 28.49 | SSIM: 0.9797 | LPIPS: 0.1817 | Color: 58.5%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0159.jpg] PSNR: 26.30 | SSIM: 0.9699 | LPIPS: 0.1462 | Color: 65.4%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0178.jpg] PSNR: 24.99 | SSIM: 0.9292 | LPIPS: 0.2499 | Color: 83.5%
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0187.jpg] PSNR: 25.16 | SSIM: 0.9584 | LPIPS: 0.2442 | Color: 67.5%
[VIDEO1_Nh

## Demo Gradio

In [2]:
!pip install gradio gradio_imageslider timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 5.0 MB/s  0:00:04m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 1.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [gradio_imageslider]radio]]]


In [14]:
!python gradio_app_WIP.py

/home/h/ddcolor312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Đang tải model...
Model đã sẵn sàng!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://83e5ac43e3f05b377e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Đang xử lý...
Xong!
Keyboard interruption in main thread... closing server.
^C
Traceback (most recent call last):
  File "/home/h/ddcolor312/lib/python3.12/site-packages/gradio/blocks.py", line 3048, in block_thread
    time.sleep(0.1)
Keyb